# 第 5 周练习解答 —— 人工智能个人知识工作者

## 练习目标

搭建一个**本地优先**的私人知识库聊天机器人：

- 上传 PDF / DOCX / TXT / MD / CSV，或从 Google Workspace 同步文档
- 用 **ChromaDB + SentenceTransformer** 向量化并持久化
- 用 **Anthropic Claude** 基于检索到的片段回答问题
- 数据默认留在本机（向量库目录 `chroma_db`、文档目录 `docs`）

## 和本课第 5 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文档解析与分块 | `parse_*` / `chunk_text` |
| 向量库 | Chroma `PersistentClient` + `all-MiniLM-L6-v2` |
| RAG 问答 | `retrieve_context` + Claude `messages.create` |
| Gradio UI | 本地文档 Tab + Google Workspace Tab |

## 怎么跑

1. 配置环境变量：`ANTHROPIC_API_KEY`（可选 Google OAuth 相关变量）
2. 按下一格安装依赖，再自上而下运行
3. 在 Gradio 里上传文档 → Index → 提问


In [ ]:
# ========== 安装依赖：Gradio UI / Chroma / 嵌入 / Claude / 办公文档解析 ==========
# 逻辑未改；首次环境或缺包时再跑。已装齐可跳过。

# gradio：界面；chromadb：向量库；sentence-transformers：嵌入模型后端
# anthropic：Claude SDK；pypdf / python-docx：PDF/Word；pandas：CSV
!pip install gradio chromadb sentence-transformers anthropic pypdf python-docx pandas


In [ ]:
# ========== 导入：本地解析 / Chroma / Claude / Google OAuth 客户端 ==========

# os：路径与环境变量
import os
# Gradio：浏览器 UI
import gradio as gr
# Chroma：持久化向量库
import chromadb
# Chroma 自带的 SentenceTransformer 嵌入函数封装
from chromadb.utils import embedding_functions
# Anthropic 官方客户端：调 Claude
from anthropic import Anthropic
# PDF 逐页抽文本
from pypdf import PdfReader
# Word .docx 读段落
from docx import Document
# CSV → 表格字符串
import pandas as pd
# Google OAuth 本地授权流
from google_auth_oauthlib.flow import InstalledAppFlow
# 刷新过期 token
from google.auth.transport.requests import Request
# 构建 Drive / Docs API 客户端
from googleapiclient.discovery import build
# 把 Google 凭据序列化到本地 pickle
import pickle
# 清空向量库目录时用
import shutil


In [ ]:
# ========== 配置：API Key / 路径 / Claude 模型 / Google OAuth ==========

# 从环境变量读 Anthropic Key（勿写进笔记本正文）
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
# Hugging Face Token（本练习主路径未强依赖，保留读取）
HF_TOKEN = os.environ.get("HF_TOKEN")
# 本地上传文档存放目录
DOCS_FOLDER = "docs"
# Chroma 持久化目录
CHROMA_FOLDER = "chroma_db"
# Claude 模型 id（可运行字符串，勿改）
MODEL = "claude-opus-4-6"

# Google OAuth 客户端三件套（环境变量）
GOOGLE_CLIENT_ID = os.environ.get("GOOGLE_CLIENT_ID")
GOOGLE_CLIENT_SECRET = os.environ.get("GOOGLE_CLIENT_SECRET")
GOOGLE_REDIRECT_URI = os.environ.get("GOOGLE_REDIRECT_URI")
# 本地缓存已授权 token 的文件名
TOKEN_FILE = "google_token.pickle"

# Drive 只读 + Docs 只读 scope（URL 保持原样）
SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/documents.readonly"
]

# InstalledAppFlow 需要的 client_config 字典结构
CLIENT_CONFIG = {
    "installed": {
        "client_id": GOOGLE_CLIENT_ID,
        "client_secret": GOOGLE_CLIENT_SECRET,
        "redirect_uris": ["http://localhost"],
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token"
    }
}

# 确保文档目录与向量库目录存在
os.makedirs(DOCS_FOLDER, exist_ok=True)
os.makedirs(CHROMA_FOLDER, exist_ok=True)

# 创建 Anthropic 客户端，后续 messages.create 都用它
client = Anthropic(api_key=ANTHROPIC_API_KEY)


In [ ]:
# ========== 向量库：SentenceTransformer 嵌入 + Chroma collection ==========

# 嵌入函数：本地 all-MiniLM-L6-v2（小而快，适合个人知识库）
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 持久化客户端：数据落在 CHROMA_FOLDER
chroma_client = chromadb.PersistentClient(path=CHROMA_FOLDER)

# 取得或创建名为 knowledge_base 的 collection，并挂上嵌入函数
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    embedding_function=embed_fn
)


In [ ]:
# ========== 文件解析器：按扩展名把文件变成纯文本 ==========

# PDF：逐页 extract_text，跳过空页，用换行拼接
def parse_pdf(filepath):
    reader = PdfReader(filepath)
    return "\n".join(page.extract_text() for page in reader.pages if page.extract_text())

# DOCX：拼接非空段落
def parse_docx(filepath):
    doc = Document(filepath)
    return "\n".join(para.text for para in doc.paragraphs if para.text.strip())

# TXT / MD：整文件 UTF-8 读入
def parse_txt(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

# CSV：pandas 读表后 to_string（不含行号索引）
def parse_csv(filepath):
    df = pd.read_csv(filepath)
    return df.to_string(index=False)

# 统一入口：按后缀选解析器；不支持则返回 None
def parse_file(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    parsers = {
        ".pdf": parse_pdf,
        ".docx": parse_docx,
        ".txt": parse_txt,
        ".md": parse_txt,
        ".csv": parse_csv
    }
    parser = parsers.get(ext)
    if parser:
        return parser(filepath)
    return None


In [ ]:
# ========== 文档索引：按词分块 + 写入 Chroma（跳过已索引） ==========

# 按「词」滑动窗口分块：chunk_size 词一段，overlap 词重叠，减轻截断丢上下文
def chunk_text(text, chunk_size=500, overlap=50):
    # 空白分词（英文知识库常用；中文可改成字/句切）
    words = text.split()
    chunks = []
    i = 0
    # 窗口前进：每次跳 chunk_size - overlap
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# 扫描 DOCS_FOLDER：解析 → 分块 → collection.add；已存在 source 则跳过
def index_documents():
    files = os.listdir(DOCS_FOLDER)
    # 空目录时的提示文案保持原样（可运行/界面字符串）
    if not files:
        return "No files found in /docs folder."

    indexed = 0
    skipped = 0

    for filename in files:
        filepath = os.path.join(DOCS_FOLDER, filename)
        text = parse_file(filepath)

        # 解析失败或不支持格式
        if not text:
            skipped += 1
            continue

        chunks = chunk_text(text)
        # 按 metadata.source == 文件名查是否已入库
        existing = collection.get(where={"source": filename})

        if existing["ids"]:
            skipped += 1
            continue

        # 稳定 id：文件名_chunk_序号；metadata 只记 source
        ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
        metadatas = [{"source": filename} for _ in chunks]

        # add 时由 embed_fn 自动算向量
        collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        indexed += 1

    return f"Indexing complete ✓ | Indexed: {indexed} files | Skipped: {skipped} files | Total chunks: {collection.count()}"


In [ ]:
# ========== 检索与问答：向量 Top-n → 拼 context → Claude 回答 ==========

# 用 query_texts 做相似度检索；返回拼接上下文 + 去重来源列表
def retrieve_context(query, n_results=5):
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    # 多段文档用空行拼接，供 system prompt 使用
    context = "\n\n".join(docs)
    unique_sources = list(set(sources))
    return context, unique_sources

# Gradio 旧版 (user, assistant) 历史元组风格的问答封装
def ask_claude(query, chat_history):
    # 先检索再问模型
    context, sources = retrieve_context(query)

    # system prompt 含检索上下文；英文指令保持原样（影响模型行为）
    system_prompt = f"""You are a helpful personal knowledge assistant. 
Answer the user's question using only the context provided from their personal documents.
If the answer is not in the context, say so clearly.

Context from personal documents:
{context}"""

    # 把历史轮次展开成 Anthropic messages 列表，再追加当前 user
    messages = []
    for user_msg, assistant_msg in chat_history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": query})

    # 调用 Claude；model / max_tokens 保持原设置
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=messages
    )

    # 取第一条 text block；若有来源则追加 Markdown 脚注
    answer = response.content[0].text
    if sources:
        answer += f"\n\n*Sources: {', '.join(sources)}*"

    return answer


In [ ]:
# ========== 知识库管理：列出已索引来源 / 一键清空 ==========

# 从 collection 取出全部 metadata，汇总去重后的 source 列表
def list_indexed_documents():
    results = collection.get()
    if not results["ids"]:
        return "No documents indexed yet."
    sources = sorted(set(m["source"] for m in results["metadatas"]))
    doc_list = "\n".join(f"• {s}" for s in sources)
    return f"Indexed documents ({len(sources)}):\n\n{doc_list}"

# 删除 collection + 清空 chroma 目录与 docs 文件，再重建空 collection
def clear_all():
    global collection

    # 删掉名为 knowledge_base 的集合
    chroma_client.delete_collection("knowledge_base")

    # 物理删除持久化目录后重建空目录
    shutil.rmtree(CHROMA_FOLDER, ignore_errors=True)
    os.makedirs(CHROMA_FOLDER, exist_ok=True)

    # 重新挂上同一 embed_fn
    collection = chroma_client.get_or_create_collection(
        name="knowledge_base",
        embedding_function=embed_fn
    )

    # 清空本地上传的原始文件
    for f in os.listdir(DOCS_FOLDER):
        os.remove(os.path.join(DOCS_FOLDER, f))

    return "All documents cleared.", list_indexed_documents()


In [ ]:
# ========== Google 认证：加载/刷新 token，构建 Drive + Docs 服务 ==========

# 返回 (drive_service, docs_service)；token 缓存在 TOKEN_FILE
def get_google_services():
    creds = None
    # 已有 pickle 则先加载
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, "rb") as f:
            creds = pickle.load(f)
    # 无效或过期：能 refresh 就 refresh，否则走本地 OAuth 浏览器流
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_config(CLIENT_CONFIG, SCOPES)
            creds = flow.run_local_server(port=0)
        # 写回 pickle，下次免登录
        with open(TOKEN_FILE, "wb") as f:
            pickle.dump(creds, f)
    # 构建两个 API 客户端
    drive_service = build("drive", "v3", credentials=creds)
    docs_service = build("docs", "v1", credentials=creds)
    return drive_service, docs_service

# 删除本地 token，下次 Sync 需重新授权
def disconnect_google():
    if os.path.exists(TOKEN_FILE):
        os.remove(TOKEN_FILE)
        return "Disconnected from Google Workspace ✓ Re-sync to reconnect."
    return "No active Google session found."


In [ ]:
# ========== Google Workspace：分页拉取 Docs/Sheets/Slides 并索引 ==========

# Google 原生 MIME → export 目标 MIME（Docs→纯文本，Sheets→CSV，Slides→纯文本）
GOOGLE_MIME_EXPORT = {
    "application/vnd.google-apps.document": "text/plain",
    "application/vnd.google-apps.spreadsheet": "text/csv",
    "application/vnd.google-apps.presentation": "text/plain"
}

# 分页游标：None 表示从第一页开始；Load More 时沿用
google_page_token = None
# 跨批次累计已成功索引的文件数（仅统计，不影响去重）
total_files_indexed = 0

# 用 Docs API 递归抽 document body 里的 textRun
def fetch_google_doc_text(docs_service, file_id):
    doc = docs_service.documents().get(documentId=file_id).execute()
    text = ""
    for element in doc.get("body", {}).get("content", []):
        paragraph = element.get("paragraph")
        if paragraph:
            for part in paragraph.get("elements", []):
                text_run = part.get("textRun")
                if text_run:
                    text += text_run.get("content", "")
    return text.strip()

# 非 Docs：用 Drive files().export 按 MIME 导出，再 decode 成字符串
def fetch_exported_text(drive_service, file_id, mime_type):
    export_mime = GOOGLE_MIME_EXPORT[mime_type]
    content = drive_service.files().export(fileId=file_id, mimeType=export_mime).execute()
    return content.decode("utf-8").strip() if isinstance(content, bytes) else content.strip()

# 核心：一页最多 20 个文件；reset=True 时从第一页重新扫
def fetch_and_index_google_workspace(reset=False):
    global google_page_token, total_files_indexed

    if reset:
        google_page_token = None
        total_files_indexed = 0

    # 授权失败直接返回状态字符串 + 当前已索引列表
    try:
        drive_service, docs_service = get_google_services()
    except Exception as e:
        return f"Auth failed: {str(e)}", list_indexed_documents()

    # Drive 查询：三种 Workspace MIME，且未进回收站
    query = " or ".join([f"mimeType='{m}'" for m in GOOGLE_MIME_EXPORT.keys()])
    query += " and trashed=false"

    try:
        response = drive_service.files().list(
            q=query,
            corpora="allDrives",
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType)",
            pageSize=20,
            pageToken=google_page_token
        ).execute()
    except Exception as e:
        return f"Drive API error: {str(e)}", list_indexed_documents()

    files = response.get("files", [])
    # 更新下一页 token（可能为 None 表示结束）
    google_page_token = response.get("nextPageToken")

    if not files:
        return "No Google Workspace files found.", list_indexed_documents()

    indexed, skipped = 0, 0

    for file in files:
        file_id = file["id"]
        mime_type = file["mimeType"]
        # 用 MIME 末段当类型标签，拼进 source 名避免冲突
        file_type = mime_type.split(".")[-1]
        doc_name = f"gdrive_{file_type}_{file['name']}"

        # 已索引过同名 source 则跳过
        existing = collection.get(where={"source": doc_name})
        if existing["ids"]:
            skipped += 1
            continue

        try:
            # Docs 走 Docs API；Sheets/Slides 走 export
            if mime_type == "application/vnd.google-apps.document":
                text = fetch_google_doc_text(docs_service, file_id)
            else:
                text = fetch_exported_text(drive_service, file_id, mime_type)
        except Exception as e:
            print(f"Skipping {file['name']}: {str(e)}")
            skipped += 1
            continue

        if not text:
            skipped += 1
            continue

        # 与本地文档同一套分块 + add
        chunks = chunk_text(text)
        ids = [f"{doc_name}_chunk_{i}" for i in range(len(chunks))]
        metadatas = [{"source": doc_name} for _ in chunks]

        collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        indexed += 1

    total_files_indexed += indexed
    more = "More files available — click Load More." if google_page_token else "All files synced ✓"
    status = f"Batch complete | This batch: {indexed} indexed, {skipped} skipped | Total indexed: {total_files_indexed} | {more}"

    return status, list_indexed_documents()

# UI：「Sync First 20」→ reset=True
def sync_google_fresh():
    return fetch_and_index_google_workspace(reset=True)

# UI：「Load More」→ 继续下一页
def load_more_google():
    return fetch_and_index_google_workspace(reset=False)


In [ ]:
# ========== Gradio 界面：本地上传索引 + Google 同步 + 双 Tab 聊天 ==========

# 上传多个文件到 DOCS_FOLDER，再调用 index_documents
def upload_and_index(files):
    if not files:
        return "No files uploaded.", list_indexed_documents()
    for file in files:
        # Gradio 临时路径 → 复制到 docs/
        filename = os.path.basename(file.name)
        dest = os.path.join(DOCS_FOLDER, filename)
        with open(file.name, "rb") as src, open(dest, "wb") as dst:
            dst.write(src.read())
    status = index_documents()
    return status, list_indexed_documents()

# 新版 Chatbot：history 是 {role, content} 字典列表
def chat(user_message, history):
    # 空输入：不调用 API，原样返回
    if not user_message.strip():
        return "", history

    # 先向量检索拿到 context / sources
    context, sources = retrieve_context(user_message)

    # 复制历史 messages，再追加当前 user
    messages = []
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": user_message})

    # system 里注入检索上下文（英文 prompt 不翻译）
    system_prompt = f"""You are a helpful personal knowledge assistant.
Answer the user's question using only the context provided from their personal documents.
If the answer is not in the context, say so clearly.

Context from personal documents:
{context}"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=messages
    )

    answer = response.content[0].text
    if sources:
        answer += f"\n\n*Sources: {', '.join(sources)}*"

    # 把本轮 user/assistant 写回 history；清空输入框
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": answer})

    return "", history

# Blocks：两个 Tab 共用同一套向量库与 chat 逻辑
with gr.Blocks(title="Personal Knowledge Worker") as app:
    gr.Markdown("# Personal Knowledge Worker")

    with gr.Tabs():
        with gr.Tab("Local Documents"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Upload Documents")
                    file_input = gr.File(file_count="multiple", label="Select Files")
                    index_btn = gr.Button("Index Documents", variant="primary")
                    index_status = gr.Textbox(label="Status", interactive=False)
                    gr.Markdown("### Knowledge Base")
                    doc_list = gr.Textbox(label="Indexed Documents", lines=6, interactive=False,
                                          value=list_indexed_documents())
                    clear_btn = gr.Button("Clear All Documents", variant="stop")

                with gr.Column(scale=2):
                    gr.Markdown("### Chat with your Knowledge Base")
                    chatbot = gr.Chatbot(height=450)
                    msg_input = gr.Textbox(placeholder="Ask a question about your documents...", label="Your Question")
                    clear_chat_btn = gr.ClearButton([msg_input, chatbot])

        with gr.Tab("Google Workspace"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Google Workspace Sync")
                    gr.Markdown("Fetches 20 files at a time from your Google Drive.")
                    with gr.Row():
                        google_sync_btn = gr.Button("Sync First 20", variant="primary")
                        google_more_btn = gr.Button("Load More", variant="secondary")
                    google_disconnect_btn = gr.Button("Disconnect Google", variant="secondary")
                    google_status = gr.Textbox(label="Status", interactive=False)
                    gr.Markdown("### Knowledge Base")
                    google_doc_list = gr.Textbox(label="Indexed Documents", lines=6, interactive=False,
                                                  value=list_indexed_documents())
                    google_clear_btn = gr.Button("Clear All Documents", variant="stop")

                with gr.Column(scale=2):
                    gr.Markdown("### Chat with your Knowledge Base")
                    chatbot2 = gr.Chatbot(height=450)
                    msg_input2 = gr.Textbox(placeholder="Ask a question about your Google Workspace...", label="Your Question")
                    clear_chat_btn2 = gr.ClearButton([msg_input2, chatbot2])

    # 本地选项卡事件：索引 / 清空 / 回车提问
    index_btn.click(upload_and_index, inputs=file_input, outputs=[index_status, doc_list])
    clear_btn.click(clear_all, outputs=[index_status, doc_list])
    msg_input.submit(chat, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])

    # Google 标签事件：首批同步 / 加载更多 / 断开 / 清空 / 提问
    google_sync_btn.click(sync_google_fresh, outputs=[google_status, google_doc_list])
    google_more_btn.click(load_more_google, outputs=[google_status, google_doc_list])
    google_disconnect_btn.click(disconnect_google, outputs=[google_status])
    google_clear_btn.click(clear_all, outputs=[google_status, google_doc_list])
    msg_input2.submit(chat, inputs=[msg_input2, chatbot2], outputs=[msg_input2, chatbot2])

# 启动 Gradio（默认本机）
app.launch()
